## Cell 1 – Install Required Libraries

In [1]:
!pip install -q pandas numpy sentence-transformers faiss-cpu chromadb kaggle datasets

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import time

from sentence_transformers import SentenceTransformer

import faiss
import chromadb

c:\Users\dell\anaconda3\Anaconda\envs\myenv6\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download Kaggle Dataset

In [3]:
!kaggle datasets download -d snap/amazon-fine-food-reviews

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0




  0%|          | 0.00/242M [00:00<?, ?B/s]
  0%|          | 1.00M/242M [00:01<05:32, 760kB/s]
  1%|          | 2.00M/242M [00:01<03:07, 1.34MB/s]
  1%|          | 3.00M/242M [00:01<02:05, 2.01MB/s]
  2%|▏         | 4.00M/242M [00:02<01:41, 2.47MB/s]
  2%|▏         | 5.00M/242M [00:02<01:25, 2.92MB/s]
  2%|▏         | 6.00M/242M [00:02<01:16, 3.23MB/s]
  3%|▎         | 7.00M/242M [00:02<01:07, 3.67MB/s]
  3%|▎         | 8.00M/242M [00:03<01:02, 3.94MB/s]
  4%|▎         | 9.00M/242M [00:03<00:58, 4.19MB/s]
  4%|▍         | 10.0M/242M [00:03<01:00, 4.04MB/s]
  5%|▍         | 11.0M/242M [00:03<01:04, 3.76MB/s]
  5%|▍         | 12.0M/242M [00:04<01:13, 3.30MB/s]
  5%|▌         | 13.0M/242M [00:04<01:07, 3.58MB/s]
  6%|▌         | 14.0M/242M [00:04<01:10, 3.40MB/s]
  6%|▌         | 15.0M/242M [00:05<01:02, 3.82MB/s]
  7%|▋         | 16.0M/242M [00:05<00:56, 4.23MB/s]
  7%|▋         | 17.0M/242M [00:05<00:59, 3.98MB/s]
  7%|▋         | 18.0M/242M [00:05<00:53, 4.40MB/s]
  8%|▊         | 19.0

In [4]:
import zipfile

with zipfile.ZipFile("amazon-fine-food-reviews.zip","r") as zip_ref:
    zip_ref.extractall()

## Load Dataset

In [5]:
df = pd.read_csv("Reviews.csv")
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## Check Dataset

In [6]:
print(df.shape)

df.info()

(568454, 10)
<class 'pandas.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Id                      568454 non-null  int64
 1   ProductId               568454 non-null  str  
 2   UserId                  568454 non-null  str  
 3   ProfileName             568428 non-null  str  
 4   HelpfulnessNumerator    568454 non-null  int64
 5   HelpfulnessDenominator  568454 non-null  int64
 6   Score                   568454 non-null  int64
 7   Time                    568454 non-null  int64
 8   Summary                 568427 non-null  str  
 9   Text                    568454 non-null  str  
dtypes: int64(5), str(5)
memory usage: 313.1 MB


## Keep Required Columns

In [7]:
df = df[['Id','Summary','Text']]

df.head()

,Id,Summary,Text
0,1,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,Cough Medicine,If you are looking for the secret ingredient i...
4,5,Great taffy,Great taffy at a great price. There was a wid...


## Handle Missing Values

In [8]:
df = df.dropna()

## Create Single Text Column

In [9]:
df["content"] = df["Summary"] + " " + df["Text"]

## Clean Text

In [10]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["content"] = df["content"].apply(clean_text)

## Use First 10,000 Records

In [11]:
df = df.iloc[:10000]

## Load Sentence Transformer

In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\dell\anaconda3\Anaconda\envs\myenv6\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## Generate Embeddings

In [13]:
start = time.time()

embeddings = model.encode(
    df["content"].tolist(),
    show_progress_bar=True
)

end = time.time()

print("Embedding Time:", end-start)

Batches: 100%|██████████| 313/313 [03:44<00:00,  1.39it/s]

Embedding Time: 224.96339082717896


## Convert to NumPy

In [14]:
embeddings = np.array(embeddings).astype("float32")

## Build FAISS Index

In [15]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

start = time.time()

index.add(embeddings)

end = time.time()

print("Indexing Time:", end-start)

Indexing Time: 0.006113767623901367


## Store Metadata

In [16]:
metadata = df.to_dict("records")

## Search Function

In [17]:
def search(query, top_k=5):

    query_embedding = model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(distances[0], indices[0]):
        results.append({
            "Id": metadata[idx]["Id"],
            "Summary": metadata[idx]["Summary"],
            "Similarity Score": float(score)
        })

    return pd.DataFrame(results)

## Test Search

In [18]:
search("healthy organic food")

,Id,Summary,Similarity Score
0,6221,Organic but not tasty!,0.643038
1,6081,NOT Organic!!!,0.694869
2,5955,"Great taste, nutritious, and organic all in one!",0.711777
3,4764,"Don't be fooled,",0.719896
4,4850,NOT organic,0.726295


## Measure Query Time

In [19]:
start = time.time()

result = search("best chocolate")

end = time.time()

print("Query Time:", end-start)

result

Query Time: 0.053502559661865234


,Id,Summary,Similarity Score
0,9885,Best of the Best,0.536516
1,5971,love it,0.558976
2,9757,Best Hot Chocolate ever,0.561366
3,1316,Much better than milk chocolate!,0.576220
4,9745,Best chocolate drink ever,0.579455


## Create ChromaDB

In [20]:
client = chromadb.Client()

collection = client.create_collection("reviews")

## Insert into ChromaDB

In [22]:
start = time.time()

batch_size = 5000

for start_idx in range(0, len(df), batch_size):
    end_idx = min(start_idx + batch_size, len(df))
    collection.add(
        ids=[str(i) for i in range(start_idx, end_idx)],
        documents=df["content"].iloc[start_idx:end_idx].tolist(),
        embeddings=embeddings[start_idx:end_idx].tolist()
    )

end = time.time()

print("ChromaDB Index Time:", end-start)

ChromaDB Index Time: 7.471624374389648


## Search ChromaDB

In [23]:
results = collection.query(
    query_embeddings=model.encode(["best chocolate"]).tolist(),
    n_results=5
)

results

{'ids': [['9884', '5970', '9756', '1315', '9744']],
 'embeddings': None,
 'documents': [['best of the best only once before have i tasted a better chocolate covered coconut macaroon and that was long ago and far away the dark chocolate is lucious and its very addicting',
   'love it some of the best chocolates i ever tasted if you are a chocolates lover then you have to try these',
   'best hot chocolate ever i purchased this for my girlfriend who loved this stuff over the xmas period now she has a supply which should last several months',
   'much better than milk chocolate we love strawberries dipped in chocolate but locally we could not find the dark chocolate type so glad we found this at amazon it is wonderful',
   'best chocolate drink ever this has a very dark chocolate flavor that mixes quickly and smoothly in hot water no chucks like most of the other brands milk chocolate is fine for those who enjoy that but for real chocolate lovers that enjoy the true flavor of dark chocola

## Compare FAISS vs ChromaDB

In [24]:
comparison = pd.DataFrame({
    "Feature":[
        "Speed",
        "Persistence",
        "Scalability",
        "Easy Setup",
        "Metadata Support"
    ],
    "FAISS":[
        "Very Fast",
        "Manual",
        "High",
        "Easy",
        "Limited"
    ],
    "ChromaDB":[
        "Fast",
        "Automatic",
        "Medium",
        "Very Easy",
        "Excellent"
    ]
})

comparison

,Feature,FAISS,ChromaDB
0,Speed,Very Fast,Fast
1,Persistence,Manual,Automatic
2,Scalability,High,Medium
3,Easy Setup,Easy,Very Easy
4,Metadata Support,Limited,Excellent


## Conclusion

In [25]:
print("""
Conclusion

1. Sentence Transformers generated semantic embeddings.
2. FAISS provided the fastest similarity search.
3. ChromaDB offered better metadata handling and persistence.
4. Both databases successfully retrieved relevant documents.
5. FAISS is preferred for high-speed search, while ChromaDB is better for production applications requiring metadata management.
""")


Conclusion

1. Sentence Transformers generated semantic embeddings.
2. FAISS provided the fastest similarity search.
3. ChromaDB offered better metadata handling and persistence.
4. Both databases successfully retrieved relevant documents.
5. FAISS is preferred for high-speed search, while ChromaDB is better for production applications requiring metadata management.

